In [ ]:
# ===== COVMEET: CoV collision search over the 124 unsolved classes — CONFIG (edit ONLY this cell) =====
# Runtime: vast.ai CPU box (no GPU needed — this pipeline is numba on cores).
# WHAT THIS RUNS: from all 124 unsolved Aut-class reps (the 39 reduced ones start at
# their CoV-descended rep), enumerate EVERY change of variables, Aut-min each output,
# pool into one store. Two seeds meeting at one orbit = the classes are STABLY
# AC-equivalent -> "124 - S remaining". Also tracked per class: any orbit BELOW the
# seed's aut-min ([DROP] lines + `improved_below_seed` in the summary).
# CoV moves only — no AC/substitution search anywhere, zero search nodes.
#
# CRASH / PREEMPT CONTRACT (the whole point of this notebook):
#   * everything lives in OUT_DIR as ONE append-only jsonl; every wave is fsynced.
#   * download OUT_DIR whenever you like (Jupyter: right-click folder -> Download).
#   * machine died? new box -> upload OUT_DIR to the same path -> Restart & Run All.
#     It repairs a torn last line, replays, and CONTINUES. Same for a plain restart.
#
# SMOKE FIRST (hard rule): leave SMOKE_RUN=True for the first execution. It runs the
# REAL pipeline at production settings, time-bounded — every row it writes is a real
# row the full run resumes from. Read the [hb]/[cum] lines (states/s, mem), THEN set
# SMOKE_RUN=False and Restart & Run All. It resumes from the smoke's rows.

OUT_DIR      = "/workspace/covmeet_out"   # << the folder you download / re-upload
SEED_SET     = "all124"                   # "all124" (the answer is 124-S) | "reduced39"
WORKERS      = "auto"                     # "auto" = cores-1, or an int; 0 = serial
WAVE         = 4096                       # states popped per wave (throughput only)
CHUNK        = 8                          # states per worker task (throughput only)
SMOKE_RUN    = True                       # True: time-bounded rehearsal, real rows
SMOKE_MINUTES = 10
MAX_HOURS    = None                       # runaway backstop only, e.g. 72; None = run
MEM_GUARD_GB = 8                          # stop cleanly if MemAvailable drops below

BRANCH     = "experiments/ppo"            # must match the git branch this file is on
REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "/workspace/ACSolverX"
UPDATE_REPO = True                        # fetch+reset on every run (hotfixes are .py-only)


In [ ]:
# ==================== SETUP (clone / reset / install / purge) ==============
# vast.ai: open this notebook in the instance's Jupyter. Everything is anchored to
# absolute paths so re-runs never nest a clone. Restart & Run All is the resume path.
import os, subprocess, sys

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])
    return p.returncode

if not os.path.isdir(os.path.dirname(REPO_DIR)):
    REPO_DIR = os.path.join(os.path.expanduser("~"), "ACSolverX")
    OUT_DIR = os.path.join(os.path.expanduser("~"), "covmeet_out")
    print("no /workspace — using", REPO_DIR, OUT_DIR)

if not os.path.isdir(REPO_DIR):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
elif UPDATE_REPO:
    sh(f"git -C {REPO_DIR} fetch origin {BRANCH}")
    sh(f"git -C {REPO_DIR} checkout {BRANCH}")
    sh(f"git -C {REPO_DIR} reset --hard origin/{BRANCH}")
print("repo at:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

try:
    import numba, numpy  # noqa
except Exception:
    sh(f"{sys.executable} -m pip install -q numba numpy")

# a fetched .py stays stale in sys.modules — purge before importing (lesson:
# git-pull-is-not-a-module-reload)
for m in [m for m in list(sys.modules) if m.split(".")[0] == "experiments"]:
    del sys.modules[m]
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.makedirs(OUT_DIR, exist_ok=True)
from experiments.stable_ac.cov.meet import covmeet
events_path, summary_path = covmeet.run_paths(OUT_DIR, SEED_SET)
if os.path.exists(events_path):
    n = sum(1 for _ in open(events_path))
    print(f"RESUME: {events_path} has {n:,} rows — the run will continue from them")
else:
    print(f"fresh run -> {events_path}")


In [ ]:
# ==================== RUN =================================================
# Restart & Run All continues: torn tail repaired, events replayed, frontier rebuilt.
# Heartbeat: [hb] every 60 s (instantaneous states/s), [cum] every ~5 min. [MERGE] and
# [DROP] lines are the two deliverables, live. Interrupting this cell is SAFE — every
# completed wave is already fsynced; run the cell again to continue.
import os
from experiments.stable_ac.cov.meet import covmeet

summary = covmeet.run(
    OUT_DIR,
    seed_set=SEED_SET,
    workers=None if WORKERS == "auto" else int(WORKERS),
    wave=int(WAVE), chunk=int(CHUNK),
    max_seconds=(SMOKE_MINUTES * 60 if SMOKE_RUN
                 else (MAX_HOURS * 3600 if MAX_HOURS else None)),
    mem_guard_gb=float(MEM_GUARD_GB),
)
print()
print("=" * 70)
print(f"classes remaining : {summary['classes_remaining']} / {summary['n_seeds']}")
print(f"merges found      : {summary['merges_found']}")
print(f"below seed aut-min: {summary['n_improved']} classes")
print(f"expanded/discover : {summary['expanded']:,} / {summary['discovered']:,}")
print(f"stopped           : {summary['stopped']}")
print(f"DOWNLOAD THIS DIR : {OUT_DIR}")
if SMOKE_RUN:
    print("SMOKE DONE — read the [hb]/[cum] rates above, then set SMOKE_RUN = False")
    print("in CONFIG and Restart & Run All. It resumes from these rows.")


In [ ]:
# ==================== STATUS / VERIFY (read-only, safe anytime) ============
# A different lifetime from the run: replays the jsonl from disk and re-derives every
# number independently, then replays a sample of recorded CoV edges through the real
# transform + Aut-min (chains verify segment by segment — never concatenated).
# Run it against a live, crashed, or finished OUT_DIR alike; it never writes.
from experiments.stable_ac.cov.meet import verify_covmeet
rc = verify_covmeet.main([OUT_DIR, "--seed-set", SEED_SET, "--sample", "1000"])
print("verify exit:", rc, "(0 = everything checked verifies)")
